# This is the master notebook for Insight Nexus (Team 6) Bank of England employer project.

In [4]:
# Import necessary libraries.
# Creating the linear regression
import numpy as np
import pandas as pd
import pylab as py
import sklearn

# Visualise the linear regression.
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as scipy
from scipy import stats
from sklearn import metrics
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.stats.api as sms
from statsmodels.graphics.gofplots import qqplot

# Import the statsmodels.
import statsmodels.api as sm 
from statsmodels.formula.api import ols

## **Preliminary sentiment analysis**

***

In [5]:
# Load the CSV file(s) as reviews.
variables = pd.read_csv('../data_clean/variables.csv')  

# Print the DataFrame.
variables.head() 

,Month / Year,Average Compound,Average Polarity,Average Subjectivity,Average lm_score,EOM,VALUES,VLOOKUP,check,UK Core inflation (%),UK Unemployment (%),UK Average Weekly earnings growth (%),UK GDP growth (%),UK benchmark rate (%),UK Economic Inactivity rate (%),UK house price index mom (%)
0,Sep-98,0.992500,0.083288,0.435206,-0.212121,Sep-98,Sep-98,Sep-98,True,1.3,6.2,NaN,0.1,7.50,23.6,0.3
1,Oct-98,0.999150,0.084071,0.381859,-0.163729,Oct-98,Oct-98,Oct-98,True,1.2,6.2,NaN,0.5,7.50,23.6,1.0
2,Nov-98,0.998033,0.081633,0.429144,-0.316236,Nov-98,Nov-98,Nov-98,True,1.1,6.1,NaN,0.2,7.25,23.5,-0.1
3,Dec-98,0.999400,0.079172,0.411767,-0.348099,Dec-98,Dec-98,Dec-98,True,1.2,6.2,NaN,0.3,6.75,23.5,-0.2
4,Jan-99,0.997600,0.086975,0.395355,-0.266017,Jan-99,Jan-99,Jan-99,True,1.2,6.2,NaN,0.2,6.25,23.3,0.7


In [6]:
# Replace the missing values with 0 (if applicable)
variables.fillna(0, inplace=True)

In [7]:
# Print the DataFrame.
variables.head() 

,Month / Year,Average Compound,Average Polarity,Average Subjectivity,Average lm_score,EOM,VALUES,VLOOKUP,check,UK Core inflation (%),UK Unemployment (%),UK Average Weekly earnings growth (%),UK GDP growth (%),UK benchmark rate (%),UK Economic Inactivity rate (%),UK house price index mom (%)
0,Sep-98,0.992500,0.083288,0.435206,-0.212121,Sep-98,Sep-98,Sep-98,True,1.3,6.2,0.0,0.1,7.50,23.6,0.3
1,Oct-98,0.999150,0.084071,0.381859,-0.163729,Oct-98,Oct-98,Oct-98,True,1.2,6.2,0.0,0.5,7.50,23.6,1.0
2,Nov-98,0.998033,0.081633,0.429144,-0.316236,Nov-98,Nov-98,Nov-98,True,1.1,6.1,0.0,0.2,7.25,23.5,-0.1
3,Dec-98,0.999400,0.079172,0.411767,-0.348099,Dec-98,Dec-98,Dec-98,True,1.2,6.2,0.0,0.3,6.75,23.5,-0.2
4,Jan-99,0.997600,0.086975,0.395355,-0.266017,Jan-99,Jan-99,Jan-99,True,1.2,6.2,0.0,0.2,6.25,23.3,0.7


In [8]:
# Determine the number of missing values.
variables.isna().sum()

Month / Year                             0
Average Compound                         0
Average Polarity                         0
Average Subjectivity                     0
Average lm_score                         0
EOM                                      0
VALUES                                   0
VLOOKUP                                  0
check                                    0
UK Core inflation (%)                    0
UK Unemployment (%)                      0
UK Average Weekly earnings growth (%)    0
UK GDP growth (%)                        0
UK benchmark rate (%)                    0
UK Economic Inactivity rate (%)          0
UK house price index mom (%)             0
dtype: int64

In [9]:
# View the metadata.
variables.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 272 entries, 0 to 271
Data columns (total 16 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Month / Year                           272 non-null    object 
 1   Average Compound                       272 non-null    float64
 2   Average Polarity                       272 non-null    float64
 3   Average Subjectivity                   272 non-null    float64
 4   Average lm_score                       272 non-null    float64
 5   EOM                                    272 non-null    object 
 6   VALUES                                 272 non-null    object 
 7   VLOOKUP                                272 non-null    object 
 8   check                                  272 non-null    object 
 9   UK Core inflation (%)                  272 non-null    float64
 10  UK Unemployment (%)                    272 non-null    float64
 11  UK Ave

In [10]:
# Basic descriptive statistics.
variables.describe()

,Average Compound,Average Polarity,Average Subjectivity,Average lm_score,UK Core inflation (%),UK Unemployment (%),UK Average Weekly earnings growth (%),UK GDP growth (%),UK benchmark rate (%),UK Economic Inactivity rate (%),UK house price index mom (%)
count,272.000000,272.000000,272.000000,272.000000,272.000000,272.000000,272.000000,272.000000,272.000000,272.000000,272.000000
mean,0.759256,0.094863,0.413387,-0.175729,1.797059,5.605147,2.722426,0.148162,2.299265,22.495956,0.497426
std,0.382848,0.016664,0.020909,0.165373,1.061534,1.378181,1.928896,1.584145,2.260670,1.618557,0.801171
min,-0.999300,0.041209,0.324390,-0.654952,-0.100000,0.000000,-2.900000,-19.200000,0.000000,0.000000,-2.500000
25%,0.603565,0.084715,0.402782,-0.276640,1.200000,4.700000,1.300000,-0.100000,0.500000,21.800000,0.000000
50%,0.995075,0.093557,0.414254,-0.191604,1.600000,5.200000,2.700000,0.200000,0.750000,22.900000,0.500000
75%,0.999243,0.103759,0.425500,-0.081818,2.100000,6.200000,4.200000,0.500000,4.562500,23.300000,1.000000
max,0.999900,0.161252,0.484238,0.321495,6.500000,8.500000,8.900000,9.300000,7.500000,23.600000,2.800000


### Remove & rename columns

In [11]:
# Drop unnecessary columns

variables.drop(columns=['Average Compound', 'Average Polarity', 'Average Subjectivity', 'EOM', 'VALUES', 'VLOOKUP', 'check'], 
               inplace=True)

# Rename the column headers

variables.rename(columns={
    'Average lm_score': 'Sentiment_LM',
    'UK Core inflation (%)' : 'Inflation_Core',
    'UK Unemployment (%)': 'Unemployment',
    'UK Average Weekly earnings growth (%)' : 'Wages_growth',
    'UK GDP growth (%)' : 'GDP_growth',
    'UK benchmark rate (%)' : 'Benchmark_rate',
    'UK Economic Inactivity rate (%)' : 'Economic_Inactivity_rate',
    'UK house price index mom (%)' : 'House_price_index'
}, inplace=True)

# View column names
variables.columns

Index(['Month / Year', 'Sentiment_LM', 'Inflation_Core', 'Unemployment',
       'Wages_growth', 'GDP_growth', 'Benchmark_rate',
       'Economic_Inactivity_rate', 'House_price_index'],
      dtype='object')

In [12]:
# View the dataframe
variables.head()

,Month / Year,Sentiment_LM,Inflation_Core,Unemployment,Wages_growth,GDP_growth,Benchmark_rate,Economic_Inactivity_rate,House_price_index
0,Sep-98,-0.212121,1.3,6.2,0.0,0.1,7.50,23.6,0.3
1,Oct-98,-0.163729,1.2,6.2,0.0,0.5,7.50,23.6,1.0
2,Nov-98,-0.316236,1.1,6.1,0.0,0.2,7.25,23.5,-0.1
3,Dec-98,-0.348099,1.2,6.2,0.0,0.3,6.75,23.5,-0.2
4,Jan-99,-0.266017,1.2,6.2,0.0,0.2,6.25,23.3,0.7


### Save the DataFrame as a CSV file

In [15]:
# Create a CSV file as output.
variables.to_csv('../data_clean/variables_clean.csv', index=False)

In [16]:
# Import new CSV file with Pandas.
variables_clean = pd.read_csv('../data_clean/variables_clean.csv')

# View DataFrame.
variables_clean.head() 

,Month / Year,Sentiment_LM,Inflation_Core,Unemployment,Wages_growth,GDP_growth,Benchmark_rate,Economic_Inactivity_rate,House_price_index
0,Sep-98,-0.212121,1.3,6.2,0.0,0.1,7.50,23.6,0.3
1,Oct-98,-0.163729,1.2,6.2,0.0,0.5,7.50,23.6,1.0
2,Nov-98,-0.316236,1.1,6.1,0.0,0.2,7.25,23.5,-0.1
3,Dec-98,-0.348099,1.2,6.2,0.0,0.3,6.75,23.5,-0.2
4,Jan-99,-0.266017,1.2,6.2,0.0,0.2,6.25,23.3,0.7


In [ ]:
# Check metadata
variables_clean.info()

In [ ]:
# Check shape
variables_clean.shape

### Determine Correlation 

In [ ]:
# Determine correlation
variables_clean.corr()

In [ ]:
# Determine correlation
corr = variables_clean.corr()

# Create the correlation heatmap

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdYlGn', center=0)
plt.title('Correlation Map (Green = Positive, Red = Negative)')
plt.show()

## Sentiment vs Inflation

In [ ]:
# Define the dependent variable.
y = variables_clean['Sentiment_LM'] 

# Define the independent variable.
x = variables_clean['Inflation_Core'] 


In [ ]:
# Visualise the data (pre-regression)
# Start with a visualisation BEFORE running linear regression.
plt.figure(figsize=(8, 4), dpi=100)
# plt.figure(dpi=120)
plt.title("Scatterplot of Sentiment versus Inflation")
plt.xlabel("Inflation_Core")
plt.ylabel("Sentiment (LM)")
sns.scatterplot(data=variables_clean,
               x= "Inflation_Core",
               y= "Sentiment_LM",
#               hue = "Color",
               palette = "dark")

plt.show()

In [ ]:
# Fit the linear model.
# Polyfit() good to use for simple linear regression (only one variable).
# Degree = 1, degree of polynomium, for SLR always 1.
reg = np.polyfit(variables_clean['Sentiment_LM'], variables_clean['Inflation_Core'], deg = 1)

# View output.
reg

In [ ]:
# Add a trendline to visualise the linear regression.
# Use the NumPy polyval method, specify the regression and the independent variable.
trend = np.polyval(reg, variables_clean['Inflation_Core'])

# View the previous scatterplot.
plt.figure(figsize=(6, 3), dpi=100)
sns.regplot(
    data=variables_clean,
    x='Inflation_Core',
    y='Sentiment_LM',
    ci=None,  # removes confidence interval
    scatter_kws={'color': 'blue', 'alpha': 0.6},
    line_kws={'color': 'red', 'linewidth': 2}
)

plt.title("Scatterplot & Line of Best Fit")
plt.xlabel("Inflation_Core")
plt.ylabel("Sentiment (LM)")
plt.show()

In [ ]:
# Pass linear regression through OLS methods.
test = ols('y ~ x', data = variables_clean).fit()

# Print the regression table.
test.summary()

## Sentiment vs Unemployment

In [ ]:
# Define the dependent variable.
y2 = variables_clean['Sentiment_LM'] 

# Define the independent variable.
x2 = variables_clean['Unemployment'] 


In [ ]:
# Visualise the data (pre-regression)
# Start with a visualisation BEFORE running linear regression.
plt.figure(figsize=(8, 4), dpi=100)
# plt.figure(dpi=120)
plt.title("Scatterplot of Sentiment versus Unemployment")
plt.xlabel("Unemployment")
plt.ylabel("Sentiment (LM)")
sns.scatterplot(data=variables_clean,
               x= "Unemployment",
               y= "Sentiment_LM",
#               hue = "Color",
               palette = "dark")

plt.show()

In [ ]:
# Fit the linear model.
# Polyfit() good to use for simple linear regression (only one variable).
# Degree = 1, degree of polynomium, for SLR always 1.
reg2 = np.polyfit(variables_clean['Sentiment_LM'], variables_clean['Unemployment'], deg = 1)

# View output.
reg2

In [ ]:
# Add a trendline to visualise the linear regression.
# Use the NumPy polyval method, specify the regression and the independent variable.
trend2 = np.polyval(reg2, variables_clean['Unemployment'])

# View the previous scatterplot.
plt.figure(figsize=(6, 3), dpi=100)
sns.regplot(
    data=variables_clean,
    x='Unemployment',
    y='Sentiment_LM',
    ci=None,  # removes confidence interval
    scatter_kws={'color': 'blue', 'alpha': 0.6},
    line_kws={'color': 'red', 'linewidth': 2}
)

plt.title("Scatterplot & Line of Best Fit")
plt.xlabel("Unemployment")
plt.ylabel("Sentiment (LM)")
plt.show()

In [ ]:
# Pass linear regression through OLS methods.
test2 = ols('y2 ~ x2', data = variables_clean).fit()

# Print the regression table.
test2.summary()

## Sentiment vs Wages Growth

In [ ]:
# Define the dependent variable.
y3 = variables_clean['Sentiment_LM'] 

# Define the independent variable.
x3 = variables_clean['Wages_growth']

In [ ]:
# Visualise the data (pre-regression)
# Start with a visualisation BEFORE running linear regression.
plt.figure(figsize=(8, 4), dpi=100)
# plt.figure(dpi=120)
plt.title("Scatterplot of Sentiment versus Wages growth")
plt.xlabel("Wages growth")
plt.ylabel("Sentiment (LM)")
sns.scatterplot(data=variables_clean,
               x= "Wages_growth",
               y= "Sentiment_LM",
#               hue = "Color",
               palette = "dark")

plt.show()

In [ ]:
# Fit the linear model.
# Polyfit() good to use for simple linear regression (only one variable).
# Degree = 1, degree of polynomium, for SLR always 1.
reg3 = np.polyfit(variables_clean['Sentiment_LM'], variables_clean['Wages_growth'], deg = 1)

# View output.
reg3

In [ ]:
# Add a trendline to visualise the linear regression.
# Use the NumPy polyval method, specify the regression and the independent variable.
trend3 = np.polyval(reg3, variables_clean['Wages_growth'])

# View the previous scatterplot.
plt.figure(figsize=(6, 3), dpi=100)
sns.regplot(
    data=variables_clean,
    x='Wages_growth',
    y='Sentiment_LM',
    ci=None,  # removes confidence interval
    scatter_kws={'color': 'blue', 'alpha': 0.6},
    line_kws={'color': 'red', 'linewidth': 2}
)

plt.title("Scatterplot & Line of Best Fit")
plt.xlabel("Wages growth")
plt.ylabel("Sentiment (LM)")
plt.show()

In [ ]:
# Pass linear regression through OLS methods.
test3 = ols('y3 ~ x3', data = variables_clean).fit()

# Print the regression table.
test3.summary()

## Sentiment vs GDP Growth

In [ ]:
# Define the dependent variable.
y4 = variables_clean['Sentiment_LM'] 

# Define the independent variable.
x4 = variables_clean['GDP_growth']

In [ ]:
# Visualise the data (pre-regression)
# Start with a visualisation BEFORE running linear regression.
plt.figure(figsize=(8, 4), dpi=100)
# plt.figure(dpi=120)
plt.title("Scatterplot of Sentiment versus GDP growth")
plt.xlabel("GDP growth")
plt.ylabel("Sentiment (LM)")
sns.scatterplot(data=variables_clean,
               x= "GDP_growth",
               y= "Sentiment_LM",
#               hue = "Color",
               palette = "dark")

plt.show()

In [ ]:
# Fit the linear model.
# Polyfit() good to use for simple linear regression (only one variable).
# Degree = 1, degree of polynomium, for SLR always 1.
reg4 = np.polyfit(variables_clean['Sentiment_LM'], variables_clean['GDP_growth'], deg = 1)

# View output.
reg4

In [ ]:
# Add a trendline to visualise the linear regression.
# Use the NumPy polyval method, specify the regression and the independent variable.
trend4 = np.polyval(reg4, variables_clean['GDP_growth'])

# View the previous scatterplot.
plt.figure(figsize=(6, 3), dpi=100)
sns.regplot(
    data=variables_clean,
    x='GDP_growth',
    y='Sentiment_LM',
    ci=None,  # removes confidence interval
    scatter_kws={'color': 'blue', 'alpha': 0.6},
    line_kws={'color': 'red', 'linewidth': 2}
)

plt.title("Scatterplot & Line of Best Fit")
plt.xlabel("GDP growth")
plt.ylabel("Sentiment (LM)")
plt.show()

In [ ]:
# Pass linear regression through OLS methods.
test4 = ols('y4 ~ x4', data = variables_clean).fit()

# Print the regression table.
test4.summary()

## Sentiment vs Benchmark rate

In [ ]:
# Define the dependent variable.
y5 = variables_clean['Sentiment_LM'] 

# Define the independent variable.
x5 = variables_clean['Benchmark_rate']

In [ ]:
# Visualise the data (pre-regression)
# Start with a visualisation BEFORE running linear regression.
plt.figure(figsize=(8, 4), dpi=100)
# plt.figure(dpi=120)
plt.title("Scatterplot of Sentiment versus Benchmark rate")
plt.xlabel("Benchmark rate")
plt.ylabel("Sentiment (LM)")
sns.scatterplot(data=variables_clean,
               x= "Benchmark_rate",
               y= "Sentiment_LM",
#               hue = "Color",
               palette = "dark")

plt.show()

In [ ]:
# Fit the linear model.
# Polyfit() good to use for simple linear regression (only one variable).
# Degree = 1, degree of polynomium, for SLR always 1.
reg5 = np.polyfit(variables_clean['Sentiment_LM'], variables_clean['Benchmark_rate'], deg = 1)

# View output.
reg5

In [ ]:
# Add a trendline to visualise the linear regression.
# Use the NumPy polyval method, specify the regression and the independent variable.
trend5 = np.polyval(reg5, variables_clean['Benchmark_rate'])

# View the previous scatterplot.
plt.figure(figsize=(6, 3), dpi=100)
sns.regplot(
    data=variables_clean,
    x='Benchmark_rate',
    y='Sentiment_LM',
    ci=None,  # removes confidence interval
    scatter_kws={'color': 'blue', 'alpha': 0.6},
    line_kws={'color': 'red', 'linewidth': 2}
)

plt.title("Scatterplot & Line of Best Fit")
plt.xlabel("Benchmark_rate")
plt.ylabel("Sentiment (LM)")
plt.show()

In [ ]:
# Pass linear regression through OLS methods.
test5 = ols('y5 ~ x5', data = variables_clean).fit()

# Print the regression table.
test5.summary()

## Sentiment vs Economic Inactivity rate

In [ ]:
# Define the dependent variable.
y6 = variables_clean['Sentiment_LM'] 

# Define the independent variable.
x6 = variables_clean['Economic_Inactivity_rate']

In [ ]:
# Visualise the data (pre-regression)
# Start with a visualisation BEFORE running linear regression.
plt.figure(figsize=(8, 4), dpi=100)
# plt.figure(dpi=120)
plt.title("Scatterplot of Sentiment versus Economic Inactivity rate")
plt.xlabel("Economic Inactivity rate")
plt.ylabel("Sentiment (LM)")
sns.scatterplot(data=variables_clean,
               x= "Economic_Inactivity_rate",
               y= "Sentiment_LM",
#               hue = "Color",
               palette = "dark")

plt.show()

In [ ]:
# Fit the linear model.
# Polyfit() good to use for simple linear regression (only one variable).
# Degree = 1, degree of polynomium, for SLR always 1.
reg6 = np.polyfit(variables_clean['Sentiment_LM'], variables_clean['Economic_Inactivity_rate'], deg = 1)

# View output.
reg6

In [ ]:
# Add a trendline to visualise the linear regression.
# Use the NumPy polyval method, specify the regression and the independent variable.
trend6 = np.polyval(reg6, variables_clean['Economic_Inactivity_rate'])

# View the previous scatterplot.
plt.figure(figsize=(6, 3), dpi=100)
sns.regplot(
    data=variables_clean,
    x='Economic_Inactivity_rate',
    y='Sentiment_LM',
    ci=None,  # removes confidence interval
    scatter_kws={'color': 'blue', 'alpha': 0.6},
    line_kws={'color': 'red', 'linewidth': 2}
)

plt.title("Scatterplot & Line of Best Fit")
plt.xlabel("Economic Inactivity rate")
plt.ylabel("Sentiment (LM)")
plt.show()

In [ ]:
# Pass linear regression through OLS methods.
test6 = ols('y6 ~ x6', data = variables_clean).fit()

# Print the regression table.
test6.summary()

## Sentiment vs House Price index

In [ ]:
# Define the dependent variable.
y7 = variables_clean['Sentiment_LM'] 

# Define the independent variable.
x7 = variables_clean['House_price_index']

In [ ]:
# Visualise the data (pre-regression)
# Start with a visualisation BEFORE running linear regression.
plt.figure(figsize=(8, 4), dpi=100)
# plt.figure(dpi=120)
plt.title("Scatterplot of Sentiment versus House Price Index")
plt.xlabel("House Price Index")
plt.ylabel("Sentiment (LM)")
sns.scatterplot(data=variables_clean,
               x= "House_price_index",
               y= "Sentiment_LM",
#               hue = "Color",
               palette = "dark")

plt.show()

In [ ]:
# Fit the linear model.
# Polyfit() good to use for simple linear regression (only one variable).
# Degree = 1, degree of polynomium, for SLR always 1.
reg7 = np.polyfit(variables_clean['Sentiment_LM'], variables_clean['House_price_index'], deg = 1)

# View output.
reg7

In [ ]:
# Add a trendline to visualise the linear regression.
# Use the NumPy polyval method, specify the regression and the independent variable.
trend7 = np.polyval(reg7, variables_clean['House_price_index'])

# View the previous scatterplot.
plt.figure(figsize=(6, 3), dpi=100)
sns.regplot(
    data=variables_clean,
    x='House_price_index',
    y='Sentiment_LM',
    ci=None,  # removes confidence interval
    scatter_kws={'color': 'blue', 'alpha': 0.6},
    line_kws={'color': 'red', 'linewidth': 2}
)

plt.title("Scatterplot & Line of Best Fit")
plt.xlabel("House_price_index")
plt.ylabel("Sentiment (LM)")
plt.show()

In [ ]:
# Pass linear regression through OLS methods.
test7 = ols('y7 ~ x7', data = variables_clean).fit()

# Print the regression table.
test7.summary()

## **FinBERT sentiment analysis**

***

## **Financial Stability Report sentiment analysis**

***

## **Monetary Policy / Inflation Report sentiment analysis**

***